# Module B1 : Qiskit — module `quantum_info`

Ce notebook accompagne les modules A2, A3 et A4.  
On utilise **exclusivement** le sous-module `qiskit.quantum_info` (pas de circuits).

**Plan :**
1. Partie 1 — Opérations sur un qubit (accompagne A2)
2. Partie 2 — Mesures sur un qubit (accompagne A3)
3. Partie 3 — Opérations sur deux qubits (accompagne A4)

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector, Operator, schmidt_decomposition
from display_utils import (
    my_display_of_tensor_product,
    my_display_of_tensor_products,
    my_display_of_schmidt_decomposition,
)

---
## Partie 1 : Opérations sur un qubit (accompagne le module A2)

Qiskit est une librairie Python créée par IBM pour implémenter et exécuter des algorithmes quantiques.

Le sous-module `quantum_info` permet de manipuler directement les **vecteurs d'état** (`Statevector`)
et les **opérateurs/matrices** (`Operator`), sans passer par la notion de circuit.

### 1.1 Définir un état (Statevector)

Un qubit dans l'état $\ket{0}$ se représente par le vecteur $(1, 0)$.

In [ ]:
# Créer |0> à partir d'un vecteur
etat_0 = Statevector([1, 0])
display(etat_0.draw('latex'))

# Créer |1> à partir d'un label
etat_1 = Statevector.from_label('1')
display(etat_1.draw('latex'))

In [ ]:
# Un état en superposition
etat_plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])
display(etat_plus.draw('latex'))

# Attention : Qiskit ne vérifie pas automatiquement la normalisation
etat_non_normalise = Statevector([1, 1])
print("Est-ce un état valide?", etat_non_normalise.is_valid())
display(etat_non_normalise.draw('latex'))

### 1.2 Définir une matrice (Operator)

Les matrices qui agissent sur les qubits s'appellent des **opérateurs**.
On peut les définir à partir d'un tableau NumPy ou d'un label Qiskit.

In [ ]:
# Matrice de Hadamard définie manuellement
H = Operator([
    [1/np.sqrt(2),  1/np.sqrt(2)],
    [1/np.sqrt(2), -1/np.sqrt(2)]
])
display(H.draw('latex'))

# Matrice de Hadamard via un label Qiskit
H2 = Operator.from_label('H')
display(H2.draw('latex'))

In [ ]:
# Matrice X (NOT quantique)
X = Operator([[0, 1], [1, 0]])
display(X.draw('latex'))

# Matrice Z
Z = Operator([[1, 0], [0, -1]])
display(Z.draw('latex'))

### 1.3 Appliquer une matrice sur un état : `evolve`

La méthode `evolve` applique un opérateur (matrice) sur un vecteur d'état :
c'est le produit matrice × vecteur.

In [ ]:
# H|0> = |+>
etat_initial = Statevector([1, 0])
etat_final = etat_initial.evolve(H)

print("H|0> =")
display(etat_final.draw('latex'))

In [ ]:
# X|0> = |1>
etat_initial = Statevector([1, 0])
etat_final = etat_initial.evolve(X)

print("X|0> =")
display(etat_final.draw('latex'))

### Exercices (Partie 1)

<mark>**Exercice 1.1**</mark> : Appliquer la porte de Hadamard sur l'état $\ket{1}$.
Observer les coefficients du nouvel état superposé.

<mark>**Exercice 1.2**</mark> : Appliquer la porte de Hadamard **puis** la porte X sur le ket $\ket{0}$.

<mark>**Exercice 1.3**</mark> : Appliquer la porte X **puis** la porte de Hadamard sur le ket $\ket{0}$.
Comparer le résultat avec l'exercice 1.2.

On remarque que **l'ordre dans lequel on applique les matrices est important** :
le produit des matrices n'est, en général, pas commutatif !

---
## Partie 2 : Mesures sur un qubit (accompagne le module A3)

Mesurer un qubit dans l'état $\ket{\psi} = \alpha\ket{0} + \beta\ket{1}$
donne $\ket{0}$ avec probabilité $|\alpha|^2$ et $\ket{1}$ avec probabilité $|\beta|^2$.

### 2.1 Probabilités de mesure

La méthode `probabilities()` calcule les probabilités de chaque résultat de mesure
dans la base computationnelle.

In [ ]:
etat = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])  # |+>
print("État :")
display(etat.draw('latex'))
print("Probabilités :", etat.probabilities())
print("→ P(|0>) =", etat.probabilities()[0], ", P(|1>) =", etat.probabilities()[1])

In [ ]:
# Un état asymétrique
etat = Statevector([np.sqrt(0.3), np.sqrt(0.7)])
print("État :")
display(etat.draw('latex'))
print("Probabilités :", etat.probabilities())

### 2.2 Simuler une mesure

La méthode `measure()` simule une mesure unique. Elle renvoie le résultat
(une chaîne '0' ou '1') et l'état après mesure.

In [ ]:
etat = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])

resultat, etat_apres = etat.measure()
print(f"Résultat de la mesure : |{resultat}>")
print("État après mesure :")
display(etat_apres.draw('latex'))

In [ ]:
# Vérification de la reproductibilité : mesurer à nouveau l'état post-mesure
resultat2, etat_apres2 = etat_apres.measure()
print(f"Deuxième mesure : |{resultat2}>  (identique au premier résultat)")
display(etat_apres2.draw('latex'))

### 2.3 Statistiques sur un grand nombre de mesures

En répétant l'expérience ("shots"), on peut vérifier que les fréquences
convergent vers les probabilités théoriques.

In [ ]:
etat = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])

# Simuler 1000 mesures
counts = etat.sample_counts(1000)
print("Résultats sur 1000 mesures :", counts)

# Affichage sous forme d'histogramme
from qiskit.visualization import plot_histogram
plot_histogram(counts)

### Exercices (Partie 2)

<mark>**Exercice 2.1**</mark> : Créer l'état $\ket{-} = \frac{1}{\sqrt{2}}(\ket{0} - \ket{1})$,
calculer ses probabilités de mesure et vérifier par simulation (1000 shots).

<mark>**Exercice 2.2**</mark> : Appliquer Z sur $\ket{+}$, puis mesurer. Les probabilités changent-elles ?

---
## Partie 3 : Opérations sur deux qubits (accompagne le module A4)

Pour un système à deux qubits, le vecteur d'état a 4 composantes
(dans la base $\{\ket{00}, \ket{01}, \ket{10}, \ket{11}\}$).

### 3.1 États à deux qubits et produit tensoriel

L'état d'un système de deux qubits indépendants est un **produit tensoriel** :
$\ket{\psi} \otimes \ket{\phi}$.

En Qiskit, on utilise la méthode `tensor` ou bien `expand`.

In [ ]:
# État |0> ⊗ |1> = |01>
q0 = Statevector.from_label('0')
q1 = Statevector.from_label('1')

etat_01 = q1.tensor(q0)   # q1 ⊗ q0 → |10> dans la convention Qiskit (qubit 0 à droite)
# Attention à la convention d'ordre de Qiskit!
print("|0>_1 ⊗ |1>_0 :")
display(etat_01.draw('latex'))

# Affichage du produit tensoriel
my_display_of_tensor_product(q1, q0)

In [ ]:
# Superposition à deux qubits : |+> ⊗ |0>
plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])
zero = Statevector.from_label('0')

etat_produit = plus.tensor(zero)
print("|+> ⊗ |0> :")
display(etat_produit.draw('latex'))

### 3.2 États de Bell et intrication

Certains états à deux qubits ne peuvent pas s'écrire comme un produit tensoriel.
Ce sont les **états intriqués**. Les plus célèbres sont les **paires de Bell**.

Par exemple : $\ket{\Phi^+} = \frac{1}{\sqrt{2}}(\ket{00} + \ket{11})$.

In [ ]:
# Créer directement un état de Bell
bell_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
print("Φ+ :")
display(bell_plus.draw('latex'))

### 3.3 Décomposition de Schmidt

La **décomposition de Schmidt** permet de déterminer si un état à deux qubits
est un état produit (séparable) ou intriqué.

- **Rang 1** : état séparable (produit)
- **Rang > 1** : état intriqué

In [ ]:
# Un état séparable : |+>|0> = (|00> + |10>)/√2
etat_separable = Statevector([1/np.sqrt(2), 0, 1/np.sqrt(2), 0])
print("État séparable :")
display(etat_separable.draw('latex'))

res = schmidt_decomposition(etat_separable, [0])
print(f"Rang de Schmidt : {len(res)}  → état produit")
my_display_of_schmidt_decomposition(res)

In [ ]:
# Un état intriqué : état de Bell Ψ- = (|01> - |10>)/√2
bell_minus = Statevector([0, 1, -1, 0] / np.sqrt(2))
print("État de Bell Ψ- :")
display(bell_minus.draw('latex'))

res = schmidt_decomposition(bell_minus, [0])
print(f"Rang de Schmidt : {len(res)}  → état intriqué!")
my_display_of_schmidt_decomposition(res)

### 3.4 Opérateurs sur deux qubits et produits tensoriels de matrices

Pour agir indépendamment sur chaque qubit, on utilise le produit tensoriel
de matrices : $(M_2 \otimes M_1) \ket{\psi}$.

In [ ]:
# H ⊗ I : Hadamard sur le qubit 1, identité sur le qubit 0
H = Operator.from_label('H')
I = Operator.from_label('I')

H_tensor_I = H.tensor(I)
print("Matrice H ⊗ I :")
display(H_tensor_I.draw('latex'))

# Appliquer H⊗I sur |00>
etat_00 = Statevector.from_label('00')
etat_final = etat_00.evolve(H_tensor_I)
print("(H⊗I)|00> :")
display(etat_final.draw('latex'))

In [ ]:
# H ⊗ H : Hadamard sur les deux qubits → superposition uniforme
H_tensor_H = H.tensor(H)
print("Matrice H ⊗ H :")
display(H_tensor_H.draw('latex'))

etat_final = etat_00.evolve(H_tensor_H)
print("(H⊗H)|00> :")
display(etat_final.draw('latex'))

### 3.5 Mesures partielles

On peut mesurer un seul qubit d'un système à deux qubits.
Pour un état intriqué, le résultat de la mesure d'un qubit
détermine l'état de l'autre.

In [ ]:
# Mesure partielle sur l'état de Bell Φ+
bell_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])

print("État initial : Φ+")
display(bell_plus.draw('latex'))

# Mesurer le qubit 0 uniquement
resultat, etat_apres = bell_plus.measure([0])
print(f"\nMesure du qubit 0 : |{resultat}>")
print("État du système après mesure :")
display(etat_apres.draw('latex'))
print("→ Les deux qubits sont parfaitement corrélés!")

### Exercices (Partie 3)

<mark>**Exercice 3.1**</mark> : Créer l'état $\ket{00} + \ket{01}$ (normalisé!).
Utiliser la décomposition de Schmidt pour déterminer s'il est intriqué ou séparable.

<mark>**Exercice 3.2**</mark> : Créer l'état de Bell $\ket{\Phi^-} = \frac{1}{\sqrt{2}}(\ket{00} - \ket{11})$.
Effectuer une mesure partielle sur le qubit 0 et observer l'état restant.

<mark>**Exercice 3.3**</mark> : Appliquer $X \otimes I$ sur l'état $\ket{00}$ et vérifier le résultat.